In [ ]:
# ==================== Cell 1: Earth Engine 初始化 ====================
import ee
ee.Authenticate()
ee.Initialize(project='the-second-project-508112')
import os
# ↑ 导入操作系统接口库
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
# 防止omp报错：OMP: Error #15

In [ ]:
# ==================== Cell 2: 参数、核心函数与处理逻辑 ====================

# ---------------- 1) 参数与核心函数 ----------------
BUFFER_SIZE = 3000
# ↑ 缓冲半径（米）。当前代码里没有实际使用，是预留参数。代码所用数据已经是正方形缓冲区了
ABS_THRESHOLD_DB = -21
# ↑ 绝对阈值（dB）。分割时像素值必须低于这个值，才可能是油膜（油膜后向散射低）！

DB_THRESHOLD_SHIFT = 3
# ↑ 相对均值偏移阈值（dB）。像素值需比局部均值低 3 dB 以上，才判定为暗斑

BRIGHT_THRESHOLD = -15
# ↑ 亮像元阈值（dB）。超过此值的像元被视为"亮目标/噪声"，在算局部均值时用均值替换掉

# 形态学处理
MORPH_RADIUS = 1
# ↑ 形态学操作的核半径（像元）。半径 1 → 3×3 方形核 → 在 10 m 分辨率下约覆盖 30 m×30 m

MIN_AREA_KM2 = 0.02
# ↑ 最小连通暗斑面积（平方公里）。小于这个值的斑块会被过滤掉，去除噪点

# lee滤波参数
LEE_KERNEL_RADIUS = 1
# ↑ Lee 滤波邻域核半径（像元）。1 → 3×3 窗口

LEE_ENL = 4
# ↑ Lee 滤波的等效视数（Equivalent Number of Looks），用于估计斑点噪声方差

# 文件夹存储路径名：'pos' 或 'neg'
SAMPLE_TAG = 'pos'
# ↑ 样本标签。'pos' 表示当前处理的是正样本（有油膜）；换 'neg' 处理负样本

# 资产路径
address = 'projects/the-second-project-508112/assets/pos2014'
# ↑ GEE 上正样本 FeatureCollection 的资产路径

fc = ee.FeatureCollection(address)
# ↑ 从 GEE 加载该 FeatureCollection（每个 Feature 是一个样本窗口，带 Date、Subcategor 等属性）

# 尺度设置
FEATURE_SCALE = 10
# ↑ 特征计算的栅格尺度（米）。10 m 对应 Sentinel-1 IW 模式像元大小

TEXTURE_SCALE = 10
# ↑ GLCM 纹理计算的尺度（米）

TEXTURE_LEVELS = 32
# ↑ GLCM 灰度量化级数。把 dB 值线性映射到 0~31 共 32 个灰度级

BACKGROUND_RING_M = 60
# ↑ 背景环的宽度（米）。围绕暗斑向外扩 60 m 作为"背景区域"

MIN_OBJECT_PIXELS = 9
# ↑ 暗斑并集至少包含 9 个像元才计算对象级特征，防止极小斑块导致统计无意义

EPS = 1e-6
# ↑ 极小值，用于防止除零

PI = 3.141592653589793
# ↑ 圆周率常量

SAFE_FILL = -999
# ↑ 缺失值填充值。当统计结果为空（如无有效像元）时用它占位

# 标签字段：正样本保留 GEE asset 的 Subcategor，负样本保留 Category。
SAMPLE_LABEL_PROPERTY = 'Subcategor' if SAMPLE_TAG == 'pos' else 'Category'
# ↑ 根据样本类型选择标签字段名：正样本用 'Subcategor'，负样本用 'Category'

# 同日多景拼接后的 SAR 有效覆盖率下限。小于该值即标记为 SAR 数据缺失，不参与分割。
SAR_COVERAGE_MIN = 0.8
# ↑ SAR 有效像元占比下限。低于 80% 就认为当天数据覆盖不全，跳过分割


def safe_name(text):
    # ↑ 定义函数：把任意字符串转成"安全文件名"
    text = str(text)
    # ↑ 转成字符串
    return ''.join(ch if ch.isalnum() or ch in ['_', '-'] else '_' for ch in text)
    # ↑ 逐字符检查：字母数字、下划线、连字符保留，其他字符一律替换成下划线


def asset_basename(asset_path):
    # ↑ 从资产路径中取最后一段作为文件名
    return safe_name(str(asset_path).split('/')[-1])
    # ↑ 用 '/' 切分，取最后一段，再做安全化处理


def short_scene_name(scene_id):
    # ↑ 把长的 Sentinel-1 scene id 缩短
    """
    eg：
    S1A_IW_GRDH_1SDV_20240130T003132_20240130T003201_052329_0653C9_9C82
    -> 0653C9_9C82

    只保留 scene_id 最后两段，文件名更短。
    如果 scene_id 格式不符合预期，则退回 safe_name(scene_id)。
    """
    # ↑ 文档字符串：说明输入输出示例
    parts = safe_name(scene_id).split('_')
    # ↑ 先安全化，再按 '_' 切分成列表
    if len(parts) >= 2:
        # ↑ 如果至少有两段
        return '_'.join(parts[-2:])
        # ↑ 取最后两段用 '_' 拼回，得到短名
    return safe_name(scene_id)
    # ↑ 否则退回完整安全名


def build_export_names(scene_id, sar_time, scene_index=None, prefix='region_scene'):
    # ↑ 构造导出文件名
    short_id = short_scene_name(scene_id)
    # ↑ 取短 scene 名
    date_part = str(sar_time)[:10].replace('-', '')
    # ↑ 取日期部分（前 10 个字符，即 YYYY-MM-DD），去掉连字符
    if scene_index is None:
        # ↑ 若未提供序号
        return f'{prefix}_{date_part}_{short_id}'
        # ↑ 返回 prefix_日期_短ID
    return f'{prefix}_{scene_index:03d}_{date_part}_{short_id}'
    # ↑ 否则返回 prefix_三位序号_日期_短ID


def _safe_num(x, default=SAFE_FILL):
    # ↑ 安全取数：x 为 null 时返回 default
    # 用 firstNonNull 兜底：x 为 null 时返回 default。
    # 之前用 ee.Algorithms.If(IsEqual(x, None), ...)，但 IsEqual(null, null) 在 GEE 中可能返回
    # null，导致 If 走 false 分支把 null 透传给 ee.Image.constant() 而报错，这里改成更稳的写法。
    return ee.Number(ee.List([x, default]).reduce(ee.Reducer.firstNonNull()))
    # ↑ 把 [x, default] 变成一个 List，用 firstNonNull reducer 取第一个非 null 的值，再转成 Number


def _safe_div(a, b, default=SAFE_FILL):
    # ↑ 安全除法：b 接近 0 时返回 default
    a = ee.Number(a)
    # ↑ 转 Number
    b = ee.Number(b)
    # ↑ 转 Number
    return ee.Number(ee.Algorithms.If(b.abs().gt(EPS), a.divide(b), default))
    # ↑ 如果 |b| > EPS，返回 a/b；否则返回 default。用 If 实现条件分支


def _safe_sqrt(x):
    # ↑ 安全开方：先取 max(0) 避免负数开方
    return ee.Number(x).max(0).sqrt()
    # ↑ x 与 0 取最大，再开方


# ---------------- Lee滤波 ----------------
# ↑ 分区注释

def lee_filter(img_db):
    # ↑ 定义 Lee 滤波函数，输入 dB 图像，输出滤波后 dB 图像
    """
    简化 Lee 滤波：功率域（线性域）局部统计 MMSE，返回 dB。
    对整个square进行lee滤波，输出波段名为VV
    """
    # ↑ 文档字符串：说明算法原理和输出
    img_db = img_db.toFloat()
    # ↑ 转成浮点型，避免整型精度问题
    img_power = ee.Image.constant(10).pow(img_db.divide(10))   # dB -> power
    # ↑ dB 转线性功率：power = 10^(dB/10)

    kernel = ee.Kernel.square(radius=LEE_KERNEL_RADIUS, units='pixels')
    # ↑ 定义方形卷积核，半径 1 像元

    # 局部一、二阶矩
    mean_power = img_power.reduceNeighborhood(ee.Reducer.mean(), kernel)
    # ↑ 邻域均值（一阶矩）
    mean_sq = img_power.pow(2).reduceNeighborhood(ee.Reducer.mean(), kernel)
    # ↑ 邻域平方均值
    var_power = mean_sq.subtract(mean_power.pow(2)).max(0)     # 总体方差
    # ↑ 方差 = E[X²] - E[X]²，取 max(0) 防止数值误差导致负值

    # 斑点噪声方差（乘性模型，强度图）
    noise_var = mean_power.pow(2).divide(LEE_ENL)
    # ↑ 斑点噪声方差估计 = 均值² / ENL
    signal_var = var_power.subtract(noise_var).max(0)
    # ↑ 信号方差 = 总方差 - 噪声方差，取 max(0)

    # Lee 权重 k
    weights = signal_var.divide(var_power.max(1e-12)).clamp(0, 1)
    # ↑ 权重 k = 信号方差/总方差，限制在 [0,1]；分母用 1e-12 防除零

    filtered_power = mean_power.add(
        weights.multiply(img_power.subtract(mean_power))
    ).max(1e-12)
    # ↑ Lee 滤波公式：滤波值 = 均值 + k × (原值 - 均值)，再取 max(1e-12) 防负数

    return filtered_power.log10().multiply(10).rename('VV')
    # ↑ 功率转回 dB：10×log10(power)，波段重命名为 'VV' 并返回


# ---------------- 风速及风向 sin 特征 ----------------
# ↑ 分区注释

def _wind_features(era5_img, region, scale=1000):
    # ↑ 从 ERA5 风场图提取区域平均风速和风向 sin 分量
    """一次区域统计输出窗口平均风速和风向单位向量的 sin 分量。"""
    # ↑ 文档字符串
    u_img = era5_img.select('u_component_of_wind_10m')
    # ↑ 选 U 分量（东西向风速）
    v_img = era5_img.select('v_component_of_wind_10m')
    # ↑ 选 V 分量（南北向风速）
    speed_img = u_img.pow(2).add(v_img.pow(2)).sqrt().rename('wind_speed')
    # ↑ 风速 = √(u² + v²)，重命名为 wind_speed

    # sin(theta) = v / speed；speed 极小时以 1e-6 防止除零。
    # 风向 sin 只在此处计算一次，随后通过 wind_dict 写入最终 Feature。
    wind_dir_sin_img = v_img.divide(speed_img.max(1e-6)).rename('wind_dir_sin')
    # ↑ 风向 sin 分量 = v / speed，分母用 max(1e-6) 防除零

    wind_stats = speed_img.addBands(wind_dir_sin_img).reduceRegion(
        reducer=ee.Reducer.mean(),
        # ↑ 用均值 reducer
        geometry=region,
        # ↑ 统计区域为样本窗口
        scale=scale,
        # ↑ 尺度 1000 m（ERA5 分辨率约 0.25°，用 1000 m 足够）
        maxPixels=1e6
        # ↑ 最多处理 100 万像元
    )
    # ↑ 对风速和风向 sin 做区域均值统计，返回 Dictionary

    return ee.Dictionary({
        'wind_mean': _safe_num(wind_stats.get('wind_speed')),
        # ↑ 平均风速，缺失时填 -999
        'wind_dir_sin': _safe_num(wind_stats.get('wind_dir_sin'), 0)
        # ↑ 风向 sin 均值，缺失时填 0
    })
    # ↑ 返回含两个键的 Dictionary


# ---------------- 单窗口阈值分割 ----------------
# ↑ 分区注释

def adaptive_threshold_window(sea_img, region, scale=10):
    # ↑ 自适应阈值分割函数：输入 Lee 滤波后的 dB 图，输出油膜掩膜
    local_dict = sea_img.reduceRegion(
        reducer=ee.Reducer.mean(),
        # ↑ 均值
        geometry=region,
        # ↑ 窗口区域
        scale=scale,
        # ↑ 尺度 10 m
        maxPixels=1e8,
        # ↑ 最大像元数
        tileScale=8
        # ↑ 瓦片缩放，缓解内存/超时问题
    )
    # ↑ 第一次区域均值统计（含亮像元）
    # 先直接算均值
    local_mean = _safe_num(local_dict.get('VV'), SAFE_FILL)
    # ↑ 取均值，缺失填 -999
    local_mean_img = ee.Image.constant(local_mean).rename('VV')
    # ↑ 把均值转成常量图像，波段名 'VV'

    # 将亮像元用均值1替换
    bright_mask = sea_img.gt(BRIGHT_THRESHOLD)
    # ↑ 亮像元掩膜：值 > -15 dB
    img_suppressed = sea_img.where(bright_mask, local_mean_img)
    # ↑ 把亮像元位置替换成均值（抑制亮目标对均值的干扰）

    local_dict2 = img_suppressed.reduceRegion(
        reducer=ee.Reducer.mean(),
        # ↑ 均值
        geometry=region,
        # ↑ 窗口区域
        scale=scale,
        # ↑ 尺度 10 m
        maxPixels=1e8,
        # ↑ 最大像元数
        tileScale=8
        # ↑ 瓦片缩放
    )
    # ↑ 第二次区域均值统计（亮像元已被替换）
    # 计算替换亮像元后的均值2
    local_mean2 = _safe_num(local_dict2.get('VV'), SAFE_FILL)
    # ↑ 取替换后的均值
    local_mean_img2 = ee.Image.constant(local_mean2).rename('VV')
    # ↑ 转成常量图像

    # 筛选像元，con1：和均值相差<db阈值(3)，con2：<abs阈值(-21)
    condition1 = img_suppressed.lt(local_mean_img2.subtract(DB_THRESHOLD_SHIFT))
    # ↑ 条件1：像素值 < 局部均值 - 3 dB（比周围暗 3 dB 以上）
    condition2 = img_suppressed.lt(ABS_THRESHOLD_DB)
    # ↑ 条件2：像素值 < -21 dB（绝对暗）
    oil_mask_raw = condition1.And(condition2).rename('oil_mask')
    # ↑ 两个条件同时满足才是候选油膜，重命名 'oil_mask'

    kernel = ee.Kernel.square(MORPH_RADIUS)
    # ↑ 形态学方形核，半径 1
    oil_mask_morph = oil_mask_raw.focal_max(kernel=kernel).focal_min(kernel=kernel)
    # ↑ 先膨胀（focal_max）再腐蚀（focal_min）= 闭运算，填补小孔洞、连接邻近斑块

    # 过滤掉面积 < MIN_AREA_KM2 的小斑块，只保留大斑块。
    patch_vectors = oil_mask_morph.selfMask().reduceToVectors(
        geometry=region,
        # ↑ 矢量化的范围
        scale=scale,
        # ↑ 尺度
        geometryType='polygon',
        # ↑ 输出多边形
        eightConnected=True,
        # ↑ 八连通（包括对角相邻）
        labelProperty='label',
        # ↑ 每个连通区打上 label
        maxPixels=1e8,
        # ↑ 最大像元数
        tileScale=8
        # ↑ 瓦片缩放
    )
    # ↑ 把掩膜栅格转成连通斑块矢量集合
    patch_vectors = patch_vectors.map(
        lambda f: ee.Feature(f).set('patch_area_km2', ee.Feature(f).geometry().area(1).divide(1e6))
    )
    # ↑ 对每个斑块计算面积（平方米→平方公里），写入属性 patch_area_km2
    big_patches = patch_vectors.filter(ee.Filter.gte('patch_area_km2', MIN_AREA_KM2))
    # ↑ 只保留面积 ≥ 0.02 km² 的大斑块
    oil_mask_final = ee.Image(0).byte().paint(big_patches, 1).And(oil_mask_morph).unmask(0).rename('oil_mask')
    # ↑ 新建全 0 图像，把大斑块涂成 1，再与原掩膜取交集，unmask(0) 把无数据变 0，重命名

    return {
        'oil_mask': oil_mask_final,
        # ↑ 最终油膜掩膜
        'img_suppressed': img_suppressed.rename('VV'),
        # ↑ 亮像元被替换后的图像
        'local_mean_before': local_mean,
        # ↑ 第一次均值
        'local_mean_after': local_mean2
        # ↑ 第二次均值
    }
    # ↑ 返回字典，供后续使用


# ---------------- 窗口级特征构建 ----------------
# ↑ 分区注释

def build_window_feature(feature, region, raw_img, filtered_img, oil_mask, wind_mean, scene_id, instrument_mode, sar_time, local_mean_before, local_mean_after):
    # ↑ 构造窗口级 Feature，包含油膜面积、比例、VV 统计等
    # oil_area_m2 是窗口内全部通过面积筛选的连通暗斑总面积。
    oil_area_m2 = _safe_num(
        ee.Image.pixelArea().rename('area').updateMask(oil_mask.eq(1)).reduceRegion(
            reducer=ee.Reducer.sum(),
            # ↑ 面积求和
            geometry=region,
            # ↑ 窗口区域
            scale=FEATURE_SCALE,
            # ↑ 尺度 10 m
            maxPixels=1e8,
            # ↑ 最大像元数
            tileScale=8
            # ↑ 瓦片缩放
        ).get('area'), 0
        # ↑ 取 'area' 键的值，缺失填 0
    )
    # ↑ 计算油膜总面积（平方米）：每个像元面积 × 掩膜，再求和

    # 窗口面积直接由 region 的平面面积获得；oil_ratio 因而是总暗斑面积占窗口面积的比例。
    window_area_m2 = _safe_num(region.area(1), 0)
    # ↑ 窗口平面面积（平方米）
    oil_ratio = _safe_div(oil_area_m2, window_area_m2, 0)
    # ↑ 油膜占比 = 油膜面积 / 窗口面积
    # 原始/Lee 滤波 VV 均只保留均值与标准差，不再计算未输出的极值。
    vv_stats = raw_img.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), '', True),
        # ↑ 均值 + 标准差组合 reducer
        geometry=region, scale=FEATURE_SCALE, maxPixels=1e8, tileScale=8
        # ↑ 区域、尺度、像元上限、瓦片缩放
    )
    # ↑ 原始 VV 的均值与标准差
    vv_filtered_stats = filtered_img.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), '', True),
        # ↑ 均值 + 标准差
        geometry=region, scale=FEATURE_SCALE, maxPixels=1e8, tileScale=8
        # ↑ 区域、尺度、像元上限、瓦片缩放
    )
    # ↑ Lee 滤波后 VV 的均值与标准差

    return ee.Feature(region).copyProperties(feature).set({
        # ↑ 用 region 建 Feature，复制原 feature 的所有属性，再 set 新属性
        SAMPLE_LABEL_PROPERTY: feature.get(SAMPLE_LABEL_PROPERTY),
        # ↑ 保留标签字段
        'scene_id': scene_id,
        # ↑ 场景 ID
        'instrument_mode': instrument_mode,
        # ↑ 仪器模式（如 IW）
        'sar_time': sar_time.format('YYYY-MM-dd HH:mm:ss'),
        # ↑ SAR 成像时间，格式化为字符串
        'wind_mean': wind_mean,
        # ↑ 平均风速
        'local_mean_before': local_mean_before,
        # ↑ 第一次局部均值
        'local_mean_after': local_mean_after,
        # ↑ 第二次局部均值
        'vv_mean': _safe_num(vv_stats.get('VV_mean')),
        # ↑ 原始 VV 均值
        'vv_stdDev': _safe_num(vv_stats.get('VV_stdDev')),
        # ↑ 原始 VV 标准差
        'vv_filtered_mean': _safe_num(vv_filtered_stats.get('VV_mean')),
        # ↑ 滤波后 VV 均值
        'vv_filtered_stdDev': _safe_num(vv_filtered_stats.get('VV_stdDev')),
        # ↑ 滤波后 VV 标准差
        'oil_area_m2': oil_area_m2,
        # ↑ 油膜面积
        'oil_ratio': oil_ratio
        # ↑ 油膜占比
    })
    # ↑ 返回构造好的 Feature


# ---------------- 全部连通暗斑联合特征 ----------------
# ↑ 分区注释

def _multi_dark_spot_features(oil_mask, region):
    # ↑ 对所有连通暗斑做矢量化，聚合几何、形状、直线度特征
    """对窗口内全部连通暗斑一次矢量化，并聚合几何、形状和直线度特征。"""
    # ↑ 文档字符串
    patches = oil_mask.selfMask().reduceToVectors(
        geometry=region, scale=FEATURE_SCALE, geometryType='polygon',
        # ↑ 范围、尺度、多边形
        eightConnected=True, labelProperty='label', maxPixels=1e8, tileScale=8
        # ↑ 八连通、标签字段、像元上限、瓦片缩放
    )
    # ↑ 把油膜掩膜转成连通斑块矢量

    def add_patch_geometry(f):
        # ↑ 内部函数：为每个斑块计算几何属性
        geom = ee.Feature(f).geometry()
        # ↑ 取斑块几何
        area_m2 = geom.area(1)
        # ↑ 面积（平方米）
        perimeter_m = geom.perimeter(1)
        # ↑ 周长（米）
        bounds = ee.List(geom.bounds(1).coordinates().get(0))
        # ↑ 外接矩形的坐标环
        p0 = ee.List(bounds.get(0))
        # ↑ 左下角点
        p1 = ee.List(bounds.get(1))
        # ↑ 右下角点
        p2 = ee.List(bounds.get(2))
        # ↑ 右上角点

        # 外接矩形的经纬度跨度换算为近似长度/宽度；L/W 越大说明斑块越狭长。
        width_m = ee.Number(p1.get(0)).subtract(ee.Number(p0.get(0))).abs().multiply(111320)
        # ↑ 经度差 × 111320 ≈ 东西向长度（米）
        height_m = ee.Number(p2.get(1)).subtract(ee.Number(p1.get(1))).abs().multiply(111320)
        # ↑ 纬度差 × 111320 ≈ 南北向长度（米）
        lwr = _safe_div(width_m.max(height_m), width_m.min(height_m), 0)
        # ↑ 长宽比 = 长边 / 短边，越大越狭长

        # 4πA/P²：圆形为 1，形状越狭长或边界越曲折，数值越接近 0。
        compactness = _safe_div(ee.Number(4).multiply(PI).multiply(area_m2), perimeter_m.pow(2), 0)
        # ↑ 紧凑度 = 4πA / P²，圆形=1，越不规则越小

        # 直线度：对斑块内像元坐标做正交线性拟合。
        # λ1、λ2 为二维坐标协方差的主/次方向方差，R²=λ1/(λ1+λ2)。
        # 接近 1 表示像元沿一条拟合直线高度集中；不依赖直线斜率，竖直斑块同样适用。
        patch_mask = ee.Image(0).byte().paint(geom, 1).selfMask()
        # ↑ 把该斑块几何画成掩膜
        # 使用 SAR 栅格投影下的像元坐标，避免经纬度坐标在不同纬度具有不同尺度。
        coord_array = ee.Image.pixelCoordinates(oil_mask.projection()).toArray().updateMask(patch_mask)
        # ↑ 生成像元坐标数组图，只保留斑块内像元
        cov_dict = coord_array.reduceRegion(
            reducer=ee.Reducer.centeredCovariance(),
            # ↑ 中心化协方差 reducer
            geometry=geom, scale=FEATURE_SCALE,
            # ↑ 几何范围、尺度
            maxPixels=1e8, tileScale=8
            # ↑ 像元上限、瓦片缩放
        )
        # ↑ 计算斑块内像元坐标的 2×2 协方差矩阵
        # 极小或退化斑块可能无法返回协方差数组，此时使用零矩阵使直线度安全回退为 0。

        default_cov = ee.Array([[0, 0], [0, 0]])
        # ↑ 默认零协方差矩阵
        cov = ee.Array(ee.List([
            cov_dict.get('array'),
            # ↑ 实际协方差
            default_cov
            # ↑ 兜底零矩阵
        ]).reduce(ee.Reducer.firstNonNull()))
        # ↑ 取第一个非 null 的协方差矩阵

        cxx = ee.Number(cov.get([0, 0]))
        # ↑ 协方差矩阵 (0,0) 元素 = x 方向方差
        cxy = ee.Number(cov.get([0, 1]))
        # ↑ (0,1) 元素 = xy 协方差
        cyy = ee.Number(cov.get([1, 1]))
        # ↑ (1,1) 元素 = y 方向方差
        trace = cxx.add(cyy).max(0)
        # ↑ 迹 = cxx + cyy，取 max(0)
        discriminant = cxx.subtract(cyy).pow(2).add(cxy.pow(2).multiply(4)).max(0).sqrt()
        # ↑ 判别式 √((cxx-cyy)² + 4cxy²)，用于求特征值
        lambda1 = trace.add(discriminant).divide(2)
        # ↑ 主特征值 λ1 = (迹 + 判别式)/2
        straightness_r2 = _safe_div(lambda1, trace, 0)
        # ↑ 直线度 R² = λ1 / 迹，接近 1 表示像元沿直线分布

        return ee.Feature(f).set({
            # ↑ 返回带新属性的 Feature
            'patch_area_m2': area_m2,
            # ↑ 斑块面积
            'patch_lwr': lwr,
            # ↑ 长宽比
            'patch_compactness': compactness,
            # ↑ 紧凑度
            'patch_straightness_r2': straightness_r2
            # ↑ 直线度
        })

    patches = patches.map(add_patch_geometry)
    # ↑ 对每个斑块应用几何属性计算
    count = patches.size()
    # ↑ 斑块数量
    area_stats = ee.Dictionary(patches.aggregate_stats('patch_area_m2'))
    # ↑ 斑块面积统计字典（mean/std/max/min 等）
    lwr_stats = ee.Dictionary(patches.aggregate_stats('patch_lwr'))
    # ↑ 长宽比统计字典
    total_area = _safe_num(patches.aggregate_sum('patch_area_m2'), 0)
    # ↑ 斑块总面积

    weighted = patches.map(
        # ↑ 对每个斑块计算"属性×面积"加权值
        lambda f: ee.Feature(f).set({
            'compactness_x_area': ee.Number(ee.Feature(f).get('patch_compactness')).multiply(ee.Number(ee.Feature(f).get('patch_area_m2'))),
            # ↑ 紧凑度 × 面积
            'straightness_x_area': ee.Number(ee.Feature(f).get('patch_straightness_r2')).multiply(ee.Number(ee.Feature(f).get('patch_area_m2')))
            # ↑ 直线度 × 面积
        })
    )
    # ↑ 用于后续面积加权平均

    return ee.Dictionary({
        # ↑ 返回聚合后的特征字典
        'dark_spot_count': count,
        # ↑ 暗斑数量
        'dark_spot_area_mean_m2': _safe_num(area_stats.get('mean'), 0),
        # ↑ 面积均值
        'dark_spot_area_std_m2': _safe_num(area_stats.get('sample_sd'), 0),
        # ↑ 面积样本标准差
        'dark_spot_lwr_max': _safe_num(lwr_stats.get('max'), 0),
        # ↑ 长宽比最大值
        'dark_spot_lwr_mean': _safe_num(lwr_stats.get('mean'), 0),
        # ↑ 长宽比均值
        'dark_spot_compactness_area_weighted': _safe_div(_safe_num(weighted.aggregate_sum('compactness_x_area'), 0), total_area, 0),
        # ↑ 面积加权紧凑度
        'dark_spot_straightness_r2_area_weighted': _safe_div(_safe_num(weighted.aggregate_sum('straightness_x_area'), 0), total_area, 0)
        # ↑ 面积加权直线度
    })


# ---------------- 全部暗斑的强度、背景与纹理特征 ----------------
# ↑ 分区注释

def add_object_features(feature, region, raw_img, oil_mask):
    # ↑ 在全部暗斑并集上计算强度、背景对比和 GLCM 纹理特征
    """在全部保留暗斑的并集上计算联合几何、强度、背景对比和纹理特征。"""
    # ↑ 文档字符串
    obj = oil_mask.eq(1).selfMask()
    # ↑ 暗斑掩膜（只保留值为 1 的像元）
    object_pixel_count = _safe_num(
        oil_mask.eq(1).reduceRegion(
            reducer=ee.Reducer.sum(),
            # ↑ 求和 = 计数
            geometry=region, scale=FEATURE_SCALE,
            # ↑ 范围、尺度
            maxPixels=1e8, tileScale=8
            # ↑ 像元上限、瓦片缩放
        ).get('oil_mask'), 0
    )
    # ↑ 暗斑像元总数
    valid_object = object_pixel_count.gte(MIN_OBJECT_PIXELS)
    # ↑ 是否 ≥ 9 个像元（有效对象）

    def when_valid_object():
        # ↑ 对象有效时的分支
        # 全部暗斑的联合几何特征：面积/形态统计来自每一个连通斑块后再聚合。
        dark_spot_dict = _multi_dark_spot_features(oil_mask, region)
        # ↑ 调用前面的联合暗斑特征函数
        obj_geom = obj.reduceToVectors(
            geometry=region, scale=FEATURE_SCALE, geometryType='polygon',
            # ↑ 范围、尺度、多边形
            eightConnected=True, labelProperty='label', maxPixels=1e8, tileScale=8
            # ↑ 八连通、标签、像元上限、瓦片缩放
        ).geometry()
        # ↑ 所有暗斑并集的外轮廓几何

        # 全部暗斑并集的原始 VV 统计；背景环围绕全部斑块，空背景时回退到整窗非暗斑区域。
        obj_stats = raw_img.updateMask(obj).reduceRegion(
            reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), '', True)
                                     .combine(ee.Reducer.minMax(), '', True),
            # ↑ 均值 + 标准差 + 最小最大
            geometry=region, scale=FEATURE_SCALE, maxPixels=1e8, tileScale=8
            # ↑ 范围、尺度、像元上限、瓦片缩放
        )
        # ↑ 暗斑内部 VV 统计
        outer_ring = obj_geom.buffer(BACKGROUND_RING_M).difference(obj_geom, 1)
        # ↑ 暗斑外扩 60 m 再减去暗斑本身 = 背景环
        bg_mask = ee.Image.constant(1).clip(region).updateMask(obj.Not())
        # ↑ 窗口内非暗斑区域掩膜（整窗背景）
        bg_stats = raw_img.updateMask(bg_mask).reduceRegion(
            reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), '', True),
            # ↑ 均值 + 标准差
            geometry=outer_ring, scale=FEATURE_SCALE, maxPixels=1e8, tileScale=8
            # ↑ 在背景环内统计
        )
        # ↑ 背景环内 VV 统计
        fallback_bg_stats = raw_img.updateMask(bg_mask).reduceRegion(
            reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), '', True),
            # ↑ 均值 + 标准差
            geometry=region, scale=FEATURE_SCALE, maxPixels=1e8, tileScale=8
            # ↑ 在整窗非暗斑区域统计（兜底）
        )
        # ↑ 兜底背景统计
        mu_sce = _safe_num(ee.List([bg_stats.get('VV_mean'), fallback_bg_stats.get('VV_mean')]).reduce(ee.Reducer.firstNonNull()))
        # ↑ 背景均值：优先背景环，缺失用整窗兜底
        sigma_sce = _safe_num(ee.List([bg_stats.get('VV_stdDev'), fallback_bg_stats.get('VV_stdDev')]).reduce(ee.Reducer.firstNonNull()))
        # ↑ 背景标准差：同上
        mu_obj = _safe_num(obj_stats.get('VV_mean'))
        # ↑ 暗斑均值
        sigma_obj = _safe_num(obj_stats.get('VV_stdDev'))
        # ↑ 暗斑标准差
        min_obj = _safe_num(obj_stats.get('VV_min'))
        # ↑ 暗斑最小值
        max_obj = _safe_num(obj_stats.get('VV_max'))
        # ↑ 暗斑最大值
        intensity_ratio = _safe_div(mu_obj, mu_sce, 0)
        # ↑ 强度比 = 暗斑均值 / 背景均值
        isdr = _safe_div(sigma_obj, sigma_sce, 0)
        # ↑ 标准差比 = 暗斑标准差 / 背景标准差
        # isri=对象均值/对象标准差，用于描述联合暗斑内部散射的相对离散程度。
        isri = _safe_div(mu_obj, sigma_obj, 0)
        # ↑ 暗斑均值/标准差

        # 在全部暗斑并集内汇总 GLCM 纹理均值。
        texture_img = raw_img.clamp(-35, 0).unitScale(-35, 0).multiply(TEXTURE_LEVELS - 1).toInt().rename('VV')
        # ↑ 把 dB 值截断到 [-35,0]，归一化到 [0,1]，乘 31 取整 → 0~31 灰度级，重命名 'VV'
        tex_stats = texture_img.glcmTexture(size=1).updateMask(obj).reduceRegion(
            reducer=ee.Reducer.mean(),
            # ↑ 均值
            geometry=region, scale=TEXTURE_SCALE,
            # ↑ 范围、纹理尺度
            maxPixels=1e8, tileScale=8
            # ↑ 像元上限、瓦片缩放
        )
        # ↑ 计算 GLCM 纹理并在暗斑内取均值
        texture_dict = ee.Dictionary({
            # ↑ 把所有纹理特征打包成字典
            'asm': _safe_num(tex_stats.get('VV_asm')),
            # ↑ 角二阶矩（能量）
            'contrast': _safe_num(tex_stats.get('VV_contrast')),
            # ↑ 对比度
            'corr': _safe_num(tex_stats.get('VV_corr')),
            # ↑ 相关性
            'var': _safe_num(tex_stats.get('VV_var')),
            # ↑ 方差
            'idm': _safe_num(tex_stats.get('VV_idm')),
            # ↑ 逆差矩
            'savg': _safe_num(tex_stats.get('VV_savg')),
            # ↑ 和平均
            'svar': _safe_num(tex_stats.get('VV_svar')),
            # ↑ 和方差
            'sentropy': _safe_num(tex_stats.get('VV_sent')),
            # ↑ 和熵
            'entropy': _safe_num(tex_stats.get('VV_ent')),
            # ↑ 熵
            'diss': _safe_num(tex_stats.get('VV_diss')),
            # ↑ 相异性
            'imcorr1': _safe_num(tex_stats.get('VV_imcorr1')),
            # ↑ 信息相关度量 1
            'imcorr2': _safe_num(tex_stats.get('VV_imcorr2')),
            # ↑ 信息相关度量 2
            'inertia': _safe_num(tex_stats.get('VV_inertia')),
            # ↑ 惯性
            'shade': _safe_num(tex_stats.get('VV_shade')),
            # ↑ 阴影
            'prom': _safe_num(tex_stats.get('VV_prom'))
            # ↑ 突出度
        })
        intensity_dict = ee.Dictionary({
            # ↑ 把强度/背景特征打包成字典
            'mu_sce': mu_sce,
            # ↑ 背景均值
            'sigma_sce': sigma_sce,
            # ↑ 背景标准差
            'mu_obj': mu_obj,
            # ↑ 暗斑均值
            'sigma_obj': sigma_obj,
            # ↑ 暗斑标准差
            'min_obj': min_obj,
            # ↑ 暗斑最小值
            'max_obj': max_obj,
            # ↑ 暗斑最大值
            'intensity_ratio': intensity_ratio,
            # ↑ 强度比
            'isdr': isdr,
            # ↑ 标准差比
            'isri': isri
            # ↑ 均值/标准差
        })
        return feature.setMulti(dark_spot_dict.combine(intensity_dict).combine(texture_dict)).set('status_34f', 'ok_34f')
        # ↑ 把三个字典合并写入 Feature，并标记状态为 ok_34f

    def when_small_object():
        # ↑ 对象太小分支
        return feature.set({'status_34f': 'object_too_small', 'object_pixel_count': object_pixel_count})
        # ↑ 标记状态为 object_too_small，并记录像元数

    return ee.Feature(ee.Algorithms.If(valid_object, when_valid_object(), when_small_object()))
    # ↑ 根据 valid_object 条件选择分支，返回 Feature


# ---------------- 单样本窗口处理：同日所有 VV Scene 拼接后再分割 ----------------
# ↑ 分区注释

def process_feature(feature):
    # ↑ 对单个样本窗口做完整处理：检索 S1 → 匹配 ERA5 → 滤波 → 分割 → 特征
    feature = ee.Feature(feature)
    # ↑ 确保是 Feature 类型
    region = feature.geometry()
    # ↑ 取窗口几何
    date_str = ee.String(feature.get('Date'))
    # ↑ 取样本日期字符串（yyyyMMdd）
    start = ee.Date.parse('yyyyMMdd', date_str)
    # ↑ 解析日期为 ee.Date
    end = start.advance(1, 'day')
    # ↑ 结束日期 = 开始 + 1 天

    s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
          .filterBounds(region).filterDate(start, end)
          .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
          .select('VV').sort('system:time_start'))
    # ↑ 加载 Sentinel-1 GRD 集合，按范围、日期过滤，只留 VV 极化，选 VV 波段，按时间排序
    image_count = s1.size()
    # ↑ 当天影像数量
    has_image = image_count.gt(0)
    # ↑ 是否有影像

    def when_has_image():
        # ↑ 有影像分支
        # 之前的逻辑：img = ee.Image(s1.first()).clip(region)，只使用当天第一景。
        # 新逻辑：拼接当天所有与窗口相交的 VV Scene，使每个像元尽可能由任一 Scene 补足。
        # 显式命名，保证覆盖率统计的字典键为 VV。
        img = s1.mosaic().clip(region).rename('VV')
        # ↑ 把所有影像拼接成一幅，裁剪到窗口，重命名 'VV'
        first_img = ee.Image(s1.first())  # 保持原 ERA5 匹配时刻：当天最早一景。
        # ↑ 取当天最早一景（用于匹配 ERA5 时刻）
        scene_ids = ee.List(s1.aggregate_array('system:index'))
        # ↑ 所有场景 ID 列表
        scene_id = ee.String(scene_ids.join('|'))
        # ↑ 用 '|' 拼成一个字符串
        instrument_mode = ee.String(first_img.get('instrumentMode'))
        # ↑ 仪器模式
        sar_time = ee.Date(first_img.get('system:time_start'))
        # ↑ SAR 成像时间

        # mosaic 掩膜在窗口内的有效像元比例；用于准确标识仍存在 SAR 空洞的 patch。
        valid_dict = img.mask().reduce(ee.Reducer.min()).rename('VV').reduceRegion(
            reducer=ee.Reducer.sum(),
            # ↑ 求和 = 有效像元计数
            geometry=region, scale=FEATURE_SCALE, maxPixels=1e8,
            # ↑ 范围、尺度、像元上限
            tileScale=8
            # ↑ 瓦片缩放
        )
        # ↑ 有效像元数
        valid_pixel_count = _safe_num(
            ee.Algorithms.If(valid_dict.contains('VV'), valid_dict.get('VV'), 0), 0
        )
        # ↑ 如果字典含 'VV' 键就取值，否则 0
        total_pixel_count_sar = _safe_num(ee.Image.constant(1).rename('constant').reduceRegion(
            reducer=ee.Reducer.count(),
            # ↑ 计数
            geometry=region, scale=FEATURE_SCALE, maxPixels=1e8,
            # ↑ 范围、尺度、像元上限
            tileScale=8
            # ↑ 瓦片缩放
        ).get('constant'), 0)
        # ↑ 窗口总像元数
        sar_coverage_ratio = _safe_div(valid_pixel_count, total_pixel_count_sar, 0)
        # ↑ SAR 覆盖率 = 有效 / 总数
        sar_missing_pixel_count = total_pixel_count_sar.subtract(valid_pixel_count).max(0)
        # ↑ 缺失像元数
        has_full_sar_coverage = sar_coverage_ratio.gte(SAR_COVERAGE_MIN)
        # ↑ 覆盖率是否 ≥ 80%

        def when_missing_coverage():
            # ↑ 覆盖率不足分支
            return ee.Feature(region).copyProperties(feature).set({
                SAMPLE_LABEL_PROPERTY: feature.get(SAMPLE_LABEL_PROPERTY),
                'image_count': image_count, 'scene_ids_used': scene_ids,
                'sar_coverage_ratio': sar_coverage_ratio,
                'sar_missing_pixel_count': sar_missing_pixel_count,
                'era5_count': 0, 'status': 'sar_missing_coverage',
                'status_34f': 'sar_missing_coverage'
            })
            # ↑ 返回标记为 sar_missing_coverage 的 Feature

        def when_full_coverage():
            # ↑ 覆盖率足够分支
            hour_start = ee.Date.fromYMD(sar_time.get('year'), sar_time.get('month'), sar_time.get('day')).advance(sar_time.get('hour'), 'hour')
            # ↑ 构造 SAR 成像时刻所在整点
            hour_end = hour_start.advance(1, 'hour')
            # ↑ 结束时刻 = 整点 + 1 小时
            era5 = ee.ImageCollection('ECMWF/ERA5/HOURLY').filterBounds(region).filterDate(hour_start, hour_end).select('u_component_of_wind_10m', 'v_component_of_wind_10m')
            # ↑ 加载 ERA5 小时数据，按范围/时刻过滤，选 U/V 风分量
            era5_count = era5.size()
            # ↑ ERA5 影像数
            has_era5 = era5_count.gt(0)
            # ↑ 是否有 ERA5 数据

            def when_has_era5():
                # ↑ 有 ERA5 分支
                # 风速均值和风向 sin 均在 _wind_features 内计算一次，并随 wind_dict 写入最终 Feature。
                wind_dict = _wind_features(ee.Image(era5.first()), region, scale=1000)
                # ↑ 计算风场特征
                wind_mean = ee.Number(wind_dict.get('wind_mean'))
                # ↑ 平均风速
                img_filtered = lee_filter(img)
                # ↑ 对拼接图做 Lee 滤波
                segment_result = adaptive_threshold_window(img_filtered, region, scale=FEATURE_SCALE)
                # ↑ 自适应阈值分割
                # oil_mask 只在此处由分割结果取得一次，后续窗口级与对象级均复用该结果。
                oil_mask = ee.Image(segment_result['oil_mask'])
                # ↑ 取油膜掩膜
                oil_sum = _safe_num(oil_mask.eq(1).reduceRegion(reducer=ee.Reducer.sum(), geometry=region, scale=FEATURE_SCALE, maxPixels=1e8).get('oil_mask'), 0)
                # ↑ 油膜像元总数
                has_oil = oil_sum.gt(0)
                # ↑ 是否有油膜

                def when_has_oil():
                    # ↑ 有油膜分支
                    base_feature = build_window_feature(feature, region, img, img_filtered, oil_mask, wind_mean, scene_id, instrument_mode, sar_time, segment_result['local_mean_before'], segment_result['local_mean_after']).set({
                        'image_count': image_count, 'scene_ids_used': scene_ids,
                        'sar_coverage_ratio': sar_coverage_ratio, 'sar_missing_pixel_count': sar_missing_pixel_count,
                        'era5_count': era5_count, 'status': 'ok'
                    }).set(wind_dict)
                    # ↑ 构造窗口级 Feature，附加元数据和风场特征
                    return add_object_features(base_feature, region, img, oil_mask)
                    # ↑ 再附加对象级特征并返回

                def when_no_oil():
                    # ↑ 无油膜分支
                    return ee.Feature(region).copyProperties(feature).set({
                        SAMPLE_LABEL_PROPERTY: feature.get(SAMPLE_LABEL_PROPERTY),
                        'image_count': image_count, 'scene_ids_used': scene_ids,
                        'sar_coverage_ratio': sar_coverage_ratio, 'sar_missing_pixel_count': sar_missing_pixel_count,
                        'era5_count': era5_count, 'status': 'no_oil', 'status_34f': 'no_oil'
                    })
                    # ↑ 返回标记 no_oil 的 Feature
                return ee.Feature(ee.Algorithms.If(has_oil, when_has_oil(), when_no_oil()))
                # ↑ 根据 has_oil 选择分支

            def when_no_era5():
                # ↑ 无 ERA5 分支
                return ee.Feature(region).copyProperties(feature).set({
                    SAMPLE_LABEL_PROPERTY: feature.get(SAMPLE_LABEL_PROPERTY),
                    'image_count': image_count, 'scene_ids_used': scene_ids,
                    'sar_coverage_ratio': sar_coverage_ratio, 'sar_missing_pixel_count': sar_missing_pixel_count,
                    'era5_count': era5_count, 'status': 'no_era5', 'status_34f': 'no_era5'
                })
                # ↑ 返回标记 no_era5 的 Feature
            return ee.Feature(ee.Algorithms.If(has_era5, when_has_era5(), when_no_era5()))
            # ↑ 根据 has_era5 选择分支

        return ee.Feature(ee.Algorithms.If(has_full_sar_coverage, when_full_coverage(), when_missing_coverage()))
        # ↑ 根据覆盖率选择分支

    def when_no_image():
        # ↑ 无 S1 影像分支
        return ee.Feature(region).copyProperties(feature).set({
            SAMPLE_LABEL_PROPERTY: feature.get(SAMPLE_LABEL_PROPERTY),
            'image_count': image_count, 'era5_count': 0,
            'sar_coverage_ratio': 0, 'sar_missing_pixel_count': 0,
            'status': 'no_s1', 'status_34f': 'no_s1'
        })
        # ↑ 返回标记 no_s1 的 Feature

    return ee.Feature(ee.Algorithms.If(has_image, when_has_image(), when_no_image()))
    # ↑ 根据是否有影像选择分支，返回最终 Feature

In [ ]:
# ==================== Cell 3: 批量处理全部样本 ====================

# ---------------- 2) 读取全部样本窗口并批量处理 ----------------
# ↑ 分区注释
# 这里已完成：S1 检索、ERA5 风场匹配、Lee 滤波、阈值分割、窗口统计和全部连通暗斑联合特征计算。
results_fc = fc.map(process_feature)
# ↑ 对 FeatureCollection 中每个样本应用 process_feature，得到结果集合

sample_count = fc.size()
# ↑ 样本总数
result_count = results_fc.size()
# ↑ 结果总数
status_hist = results_fc.aggregate_histogram('status')
# ↑ 'status' 字段的直方图（统计各状态数量）
status_34f_hist = results_fc.aggregate_histogram('status_34f')
# ↑ 'status_34f' 字段的直方图

In [ ]:
# ==================== Cell 4: 导出全部样本分割结果与基础信息 ====================

# ---------------- 3) 导出全部样本分割结果与基础信息 ----------------
# ↑ 分区注释
# results_fc 中包含：
# 1) 分割基础统计与窗口级 oil 特征；
# 2) status / status_34f；
# 3) 对于有效暗斑样本，已附加全部连通暗斑的联合几何、强度、背景和纹理特征。

sample_name = asset_basename(address)
# ↑ 从资产路径取样本名（如 pos2014）
export_folder = f'{SAMPLE_TAG}_info_{sample_name}'
# ↑ 导出文件夹名，如 pos_info_pos2014
table_prefix = f'{SAMPLE_TAG}_info_{sample_name}'
# ↑ 导出文件名前缀

task = ee.batch.Export.table.toDrive(
    # ↑ 创建导出任务：把 FeatureCollection 导出到 Google Drive
    collection=results_fc,
    # ↑ 要导出的集合
    description=table_prefix,
    # ↑ 任务描述
    folder=export_folder,
    # ↑ Drive 中的文件夹
    fileNamePrefix=table_prefix,
    # ↑ 文件名前缀
    fileFormat='CSV'
    # ↑ 导出格式为 CSV
)

task.start()
# ↑ 启动导出任务
print('Export started:', table_prefix)
# ↑ 打印提示
print('folder:', export_folder)
# ↑ 打印文件夹
print('collection: results_fc')
# ↑ 打印集合名

In [ ]:
# ==================== Cell 5: 导出机器学习训练特征样本 ====================

# ---------------- 4) 导出可用于机器学习训练的联合暗斑特征样本 ----------------
# ↑ 分区注释
# 不重新分割，直接从 results_fc 中筛选已得到有效联合暗斑特征的样本。
# 保留 status_34f 字段名仅为兼容原有筛选与导出流程；其值 ok_34f 现表示联合暗斑特征计算成功。

sample_name = asset_basename(address)
# ↑ 取样本名
export_folder_34 = f'{SAMPLE_TAG}_features_{sample_name}'
# ↑ 导出文件夹名，如 pos_features_pos2014
EXPORT_DESC_34 = f'{SAMPLE_TAG}_features_{sample_name}'
# ↑ 任务描述
EXPORT_PREFIX_34 = f'{SAMPLE_TAG}_features_{sample_name}'
# ↑ 文件名前缀

training_fc_34 = results_fc.filter(ee.Filter.eq('status_34f', 'ok_34f'))
# ↑ 只保留 status_34f == 'ok_34f' 的样本（即成功计算出联合暗斑特征）

csv_task_34 = ee.batch.Export.table.toDrive(
    # ↑ 创建导出任务
    collection=training_fc_34,
    # ↑ 要导出的集合
    description=EXPORT_DESC_34,
    # ↑ 任务描述
    folder=export_folder_34,
    # ↑ Drive 文件夹
    fileNamePrefix=EXPORT_PREFIX_34,
    # ↑ 文件名前缀
    fileFormat='CSV'
    # ↑ CSV 格式
)

csv_task_34.start()
# ↑ 启动导出任务
print('Joint-dark-spot feature CSV export task submitted.')
# ↑ 打印提示
print('folder:', export_folder_34)
# ↑ 打印文件夹
print('collection: training_fc_34')
# ↑ 打印集合名